In [ ]:
# @title Setup and Configuration

%pip install opencv-python

from google.colab import drive
import os

REPO_URL = "https://github.com/AML-Semantic-Correspondence/Semantic_Correspondence.git"

# Setting up repository...
print("\n Setting up repository...")

# First ensure we're in a safe directory
%cd /content

# Clean up any existing problematic directories
if os.path.exists('/content/Semantic_Correspondence'):
    print("Removing existing Semantic_Correspondence directory...")
    !rm -rf /content/Semantic_Correspondence

if os.path.exists('/content/semantic-correspondence'):
    print("Removing existing semantic-correspondence directory...")
    !rm -rf /content/semantic-correspondence

# Clone the repository
print("Cloning repository fresh...")
try:
    !git clone {REPO_URL}
    
    # The repo will be cloned as 'Semantic_Correspondence', let's rename it for consistency
    if os.path.exists('/content/Semantic_Correspondence'):
        !mv /content/Semantic_Correspondence /content/semantic-correspondence
        print(" Repository cloned and renamed successfully")
    else:
        print(" Repository clone failed")
except Exception as e:
    print(f" Error during clone: {e}")

# Mount drive and extract datasets
drive.mount("/content/drive", force_remount=True)
!tar -xzf "/content/drive/MyDrive/AML-Semantic-Correspondence/datasets/SPair-71k.tar.gz"
!unzip -o -q "/content/drive/MyDrive/AML-Semantic-Correspondence/datasets/PF-dataset-PASCAL.zip"
!unzip -o -q "/content/drive/MyDrive/AML-Semantic-Correspondence/datasets/PF-dataset.zip"

# Add the repository to path
%cd /content/semantic-correspondence
import sys
sys.path.append('/content/semantic-correspondence')

# Import from src modules using importlib (to handle hyphenated filenames)
import importlib.util

# Load pf-willow module
spec_willow = importlib.util.spec_from_file_location("pf_willow", "/content/semantic-correspondence/src/dataset/pf-willow.py")
pf_willow = importlib.util.module_from_spec(spec_willow)
spec_willow.loader.exec_module(pf_willow)
Dataset_pf_willow = pf_willow.Dataset_pf_willow
WILLOW_CONFIG = pf_willow.CONFIGURATION_DS

# Load pf-pascal module
spec_pascal = importlib.util.spec_from_file_location("pf_pascal", "/content/semantic-correspondence/src/dataset/pf-pascal.py")
pf_pascal = importlib.util.module_from_spec(spec_pascal)
spec_pascal.loader.exec_module(pf_pascal)
Dataset_pf_pascal = pf_pascal.Dataset_pf_pascal
PASCAL_CONFIG = pf_pascal.CONFIGURATION_DS

# Load spair71k module
spec_spair = importlib.util.spec_from_file_location("spair71k", "/content/semantic-correspondence/src/dataset/spair71k.py")
spair71k = importlib.util.module_from_spec(spec_spair)
spec_spair.loader.exec_module(spair71k)
Dataset_spair71k = spair71k.Dataset_spair71k
SPAIR_CONFIG = spair71k.CONFIGURATION_DS

# Standard imports
import random
import cv2
import matplotlib.pyplot as plt
import numpy as np
from torch.utils.data import DataLoader

print("Setup complete! Ready to visualize keypoints.")

In [ ]:
# @title Visualize PF-Willow Dataset

# Create PF-Willow dataset and select random sample
dataset = Dataset_pf_willow()
print(f"PF-Willow dataset contains {len(dataset)} image pairs")

# Select a random index
random_idx = random.randint(0, len(dataset) - 1)
print(f"Selected random image pair index: {random_idx}")

# Get the item from dataset
item = dataset[random_idx]
src_path = item["src_path"]
trg_path = item["trg_path"]
src_kps = item["src_kps"]
trg_kps = item["trg_kps"]
trg_bndbox = item["trg_bndbox"]

print(f"Source image: {src_path}")
print(f"Target image: {trg_path}")
print(f"Number of keypoints: {len(src_kps)}")

# Load images and convert them from BGR to RGB
src_img = cv2.imread(src_path)
if src_img is not None:
    src_img = cv2.cvtColor(src_img, cv2.COLOR_BGR2RGB)
    annotated_src_img = np.copy(src_img)

    # Draw keypoints on source image
    for kpt in src_kps:
        center_coordinates = (int(kpt[0]), int(kpt[1]))
        cv2.circle(annotated_src_img, center_coordinates, 3, (255, 0, 0), -1)

trg_img = cv2.imread(trg_path)
if trg_img is not None:
    trg_img = cv2.cvtColor(trg_img, cv2.COLOR_BGR2RGB)
    annotated_trg_img = np.copy(trg_img)

    # Draw bounding box on target image
    start_point = (int(trg_bndbox[0]), int(trg_bndbox[1]))
    end_point = (int(trg_bndbox[2]), int(trg_bndbox[3]))
    cv2.rectangle(annotated_trg_img, start_point, end_point, (0, 255, 0), 2)

    # Draw keypoints on target image
    for kpt in trg_kps:
        center_coordinates = (int(kpt[0]), int(kpt[1]))
        cv2.circle(annotated_trg_img, center_coordinates, 3, (255, 0, 0), -1)

    # Display the annotated images side-by-side
    plt.figure(figsize=(20, 8))

    plt.subplot(1, 2, 1)
    plt.imshow(annotated_src_img)
    plt.title('Source Image with Ground Truth Keypoints')
    plt.axis('off')

    plt.subplot(1, 2, 2)
    plt.imshow(annotated_trg_img)
    plt.title('Target Image with Ground Truth Keypoints & Bounding Box')
    plt.axis('off')

    plt.show()
else:
    print("Could not load images. Make sure datasets are properly extracted.")

In [ ]:
# @title Visualize SPair-71k Dataset with Tolerance Radii

# Create SPair-71k validation dataset
dataset = Dataset_spair71k(SPAIR_CONFIG["ALL_VAL_PATH"], SPAIR_CONFIG["PATH_VAL"])
print(f"SPair-71k validation dataset contains {len(dataset)} image pairs")

# Select a random index
random_idx = random.randint(0, len(dataset) - 1)
print(f"Selected random image pair index: {random_idx}")

# Get the item from dataset
item = dataset[random_idx]
trg_path = item["trg_path"]
trg_kps = item["trg_kps"]
trg_bndbox = item["trg_bndbox"]

print(f"Target image: {trg_path}")
print(f"Number of keypoints: {len(trg_kps)}")

# Load the target image and convert it from BGR to RGB
img = cv2.imread(trg_path)
if img is not None:
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    annotated_img = np.copy(img)

    # Choose a random keypoint from the target keypoints
    if len(trg_kps) > 0:
        chosen_kpt_idx = random.randint(0, len(trg_kps) - 1)
        chosen_kpt = trg_kps[chosen_kpt_idx]
        center_coordinates = (int(chosen_kpt[0]), int(chosen_kpt[1]))

        print(f"Selected keypoint {chosen_kpt_idx} at coordinates: {center_coordinates}")

        # Draw the chosen keypoint in RED
        cv2.circle(annotated_img, center_coordinates, 5, (255, 0, 0), -1)

        # Calculate normalization factor from target bounding box
        max_dim = max(trg_bndbox[2]-trg_bndbox[0], trg_bndbox[3]-trg_bndbox[1])
        print(f"Bounding box max dimension: {max_dim:.2f}")

        # Define tolerance thresholds (PCK evaluation thresholds)
        tolerances = [0.05, 0.1, 0.2]
        colors = [(0, 0, 255), (255, 255, 0), (255, 0, 255)]  # Blue, Yellow, Magenta
        angles = [-90, -135, -45]  # Different angles for each label

        for i, alpha in enumerate(tolerances):
            radius = int(alpha * max_dim)
            if radius == 0 and max_dim > 0: 
                radius = 1  # Ensure visibility
            
            # Draw circle
            cv2.circle(annotated_img, center_coordinates, radius, colors[i], 2)
            
            # Calculate label position
            angle_rad = np.deg2rad(angles[i])
            text_offset = 15
            text_x = int(center_coordinates[0] + (radius + text_offset) * np.cos(angle_rad))
            text_y = int(center_coordinates[1] + (radius + text_offset) * np.sin(angle_rad))
            
            # Draw white background for text
            font = cv2.FONT_HERSHEY_SIMPLEX
            font_scale = 0.6
            font_thickness = 1
            text_size = cv2.getTextSize(f"α={alpha}", font, font_scale, font_thickness)[0]
            
            padding = 3
            bg_top_left = (text_x - padding, text_y - text_size[1] - padding)
            bg_bottom_right = (text_x + text_size[0] + padding, text_y + padding)
            
            cv2.rectangle(annotated_img, bg_top_left, bg_bottom_right, (255, 255, 255), -1)
            cv2.putText(annotated_img, f"α={alpha}", (text_x, text_y), font, font_scale, (0, 0, 0), font_thickness, cv2.LINE_AA)

        # Display the annotated image
        plt.figure(figsize=(12, 10))
        plt.imshow(annotated_img)
        plt.title(f'SPair-71k Target Image with Keypoint and PCK Tolerance Radii\nKeypoint at {center_coordinates}, Bounding Box Max Dimension: {max_dim:.2f}px')
        plt.axis('off')
        plt.show()

    else:
        print("No keypoints found for the selected image pair.")
else:
    print("Could not load target image. Make sure datasets are properly extracted.")